# Layer 0 -> 1: Data Adapter & Batch Simulator (Ready for Controlled Intake)

This notebook converts 3 Vietnamese academic datasets from Hugging Face (`CSConDa`, `UIT-VSFC`, `UIT-ViSFD`) into **4 RFP-compliant input files** for Layer 1 (`Controlled Intake Engine`):

1. `service_case_feedback.csv` — Service incidents and technical support records (`DAT 01`, `DAT 04`, `Annex C`).
2. `curated_correspondence.csv` — B2B email correspondence with simulated greetings, PII signatures, and quoted history (`DAT 03`, `SEC 03`, `Annex C`).
3. `survey_responses.csv` — Survey responses with cleaned comments and 0-10 NPS ratings (`Annex C R06`).
4. `batch_manifest.json` — Batch control metadata (`expected_rows`, `source_version`, `load_mode`, `cutoff_time`, `extraction_time`, `deletion_tombstones`) (`DAT 02`, `DAT 07`, `DAT 08`).

## Module 0: Environment & Dependencies

Install `datasets<3.0.0`, `huggingface_hub`, and `pandas` for Google Colab or local execution.

In [ ]:
!pip install -q "datasets<3.0.0" huggingface_hub pandas

## Module 1: Configuration & Batch Parameters

Initialize deterministic random seed (`42`), output directory (`BASE_DIR` compatible with Google Colab `/content` and local environments), and batch simulation flags (`BATCH_ID`, `SOURCE_VERSION`, `LOAD_MODE`, `B2B_WRAP_RATIO`, `ENABLE_EDGE_CASES`).

In [ ]:
import ast
import json
import os
import random
from datetime import datetime, timedelta
import pandas as pd
from datasets import load_dataset, concatenate_datasets

random.seed(42)

BASE_DIR = "/content/ready_for_intake" if os.path.exists("/content") else "./ready_for_intake"
os.makedirs(BASE_DIR, exist_ok=True)

BATCH_ID = "BATCH_20261120_01"
SOURCE_VERSION = "v1.0.0-hf-adapter"
LOAD_MODE = "snapshot"
ENABLE_B2B_EMAIL_WRAPPER = True
B2B_WRAP_RATIO = 0.25
ENABLE_EDGE_CASES = True
BASE_TIME = datetime(2026, 11, 1, 8, 0, 0)

print(f"Ready output directory: {BASE_DIR}")

## Module 2: Hugging Face Authentication & Dataset Loader

Authenticate with `HF_TOKEN` from Colab `userdata` (if available) and load all splits of the 3 source datasets:
* `ura-hcmut/Vietnamese-Customer-Support-QA` (`CSConDa`)
* `uitnlp/vietnamese_students_feedback` (`UIT-VSFC`)
* `visolex/ViSFD` (`UIT-ViSFD`)

In [ ]:
hf_token = None
try:
    from google.colab import userdata
    from huggingface_hub import login
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Authenticated with Hugging Face Hub successfully.")
except Exception as e:
    print(f"HF_TOKEN notice: {e}")

def load_all_splits(repo_id, token=None, trust_remote=False):
    kwargs = {}
    if token:
        kwargs["token"] = token
    if trust_remote:
        kwargs["trust_remote_code"] = True
    ds_dict = load_dataset(repo_id, **kwargs)
    ds = concatenate_datasets(list(ds_dict.values()))
    print(f"Loaded {repo_id}: {len(ds)} rows across {list(ds_dict.keys())}")
    return ds

ds_csconda = load_all_splits("ura-hcmut/Vietnamese-Customer-Support-QA", token=hf_token)
ds_vsfc = load_all_splits("uitnlp/vietnamese_students_feedback", trust_remote=True)
ds_visfd = load_all_splits("visolex/ViSFD")

## Module 3: Text Extraction & B2B Email Wrapper Helpers

Define reusable transformation helpers:
* `wrap_b2b_email(core_text, idx)`: Wraps raw feedback in B2B email greetings, PII signatures (phone/email for `SEC 03` redaction testing), and `-----Original Message-----` quoted history (`DAT 03`).
* `parse_visfd_row(row)`: Extracts clean `comment` and `n_star` fields from `UIT-ViSFD` rows.

In [ ]:
def wrap_b2b_email(core_text, idx):
    contact_names = ["Nguyễn Văn An", "Trần Thị Bình", "Lê Minh Cường", "Phạm Thu Hà"]
    name = contact_names[idx % len(contact_names)]
    phone = f"09{random.randint(10000000, 99999999)}"
    email = f"lienhe{idx % 50}@doanhnghiep-khachhang.vn"
    quoted_block = (
        "\n\n-----Original Message-----\n"
        "From: Bộ phận Kinh doanh SongNhat <sales@songnhat.vn>\n"
        "Sent: 2026-10-30 14:20\n"
        "Subject: Re: Báo giá phụ tùng và lịch bảo trì định kỳ\n\n"
        "Kính gửi Quý khách hàng, vui lòng phản hồi xác nhận tiến độ giao hàng đợt trước."
    )
    signature = f"\n\nTrân trọng,\n{name}\nSĐT: {phone} | Email: {email}"
    return f"Kính gửi Bộ phận CSKH SongNhat,\n\n{core_text}{signature}{quoted_block}"

def parse_visfd_row(row):
    if isinstance(row, dict) and "comment" in row:
        return str(row.get("comment", "")).strip(), row.get("n_star")
    raw_text = str(row.get("text") or row.get("sentence") or "").strip()
    if raw_text.startswith("{") and "'comment'" in raw_text:
        try:
            parsed = ast.literal_eval(raw_text)
            if isinstance(parsed, dict):
                return str(parsed.get("comment", "")).strip(), parsed.get("n_star")
        except Exception:
            pass
    return raw_text, row.get("n_star")

## Module 4: Build `service_case_feedback.csv` (`CSConDa`)

Map `CSConDa` records to the `service_case_feedback.csv` contract with immutable `record_id`, `case_id`, `customer_id`, `channel` (`ticket`, `service_sheet`, `portal`), ISO 8601 `event_time`, `author_role` (`customer` / `staff_note`), and `product_id`.

In [ ]:
sc_records = []
for i, row in enumerate(ds_csconda):
    text_val = row.get("question") or row.get("text") or ""
    if not str(text_val).strip():
        continue
    sc_records.append({
        "record_id": f"REC_SC_{i+1:05d}",
        "case_id": f"CASE_{(i // 2) + 1:05d}",
        "customer_id": f"CUST_{random.randint(100, 150)}",
        "channel": random.choice(["ticket", "service_sheet", "portal"]),
        "event_time": (BASE_TIME + timedelta(minutes=i * 10)).isoformat(),
        "author_role": "customer" if i % 4 != 0 else "staff_note",
        "text": str(text_val).strip(),
        "product_id": f"PROD_{random.randint(1, 20):02d}",
    })

print(f"Prepared service_case_feedback rows: {len(sc_records)}")

## Module 5: Build `curated_correspondence.csv` (`UIT-VSFC`)

Map `UIT-VSFC` feedback to `curated_correspondence.csv` (`channel = "email"`), wrapping `B2B_WRAP_RATIO` (25%) of records with B2B email headers, PII signatures, and `Re:` / `Fwd:` prefixes.

In [ ]:
corr_records = []
for i, row in enumerate(ds_vsfc):
    core_text = str(row.get("sentence") or row.get("text") or "").strip()
    if not core_text:
        continue
    is_wrapped = ENABLE_B2B_EMAIL_WRAPPER and (random.random() < B2B_WRAP_RATIO)
    text_val = wrap_b2b_email(core_text, i) if is_wrapped else core_text
    prefix = random.choice(["Re: ", "Fwd: ", ""]) if is_wrapped else ""
    corr_records.append({
        "record_id": f"REC_CORR_{i+1:05d}",
        "case_id": f"CASE_CORR_{(i // 3) + 1:05d}",
        "customer_id": f"CUST_{random.randint(100, 150)}",
        "channel": "email",
        "event_time": (BASE_TIME + timedelta(minutes=i * 15)).isoformat(),
        "author_role": "customer",
        "text": text_val,
        "subject": f"{prefix}Trao đổi về báo giá và tiến độ giao linh kiện đợt {(i % 10) + 1}",
    })

print(f"Prepared curated_correspondence rows: {len(corr_records)}")

## Module 6: Build `survey_responses.csv` (`UIT-ViSFD`)

Extract clean survey comments and normalize `n_star` onto the 0-10 NPS scale (`Annex C R06`).

In [ ]:
srv_records = []
for i, row in enumerate(ds_visfd):
    clean_text, n_star = parse_visfd_row(row)
    if not clean_text:
        continue
    rating = min(10, max(0, int(n_star) * 2)) if isinstance(n_star, (int, float)) else random.randint(0, 10)
    srv_records.append({
        "record_id": f"REC_SRV_{i+1:05d}",
        "customer_id": f"CUST_{random.randint(100, 150)}",
        "channel": "survey",
        "event_time": (BASE_TIME + timedelta(minutes=i * 20)).isoformat(),
        "rating": rating,
        "text": clean_text,
    })

print(f"Prepared survey_responses rows: {len(srv_records)}")

## Module 7: Edge-Case Injection — Group 1 (Language, Quoted History & Oversized Text)

Inject 10 `VN 02` Group 1 scenarios (`DAT 03`, `ANL 03`):
* Cases 1-5: Mixed sentiment with B2B quoted history.
* Cases 6-10: Oversized `service_sheet` texts (> 8,000 characters) to verify quarantine routing without silent truncation.

In [ ]:
if ENABLE_EDGE_CASES:
    for k in range(1, 11):
        if k <= 5:
            corr_records.append({
                "record_id": f"REC_VN02_G1_{k:02d}",
                "case_id": f"CASE_VN02_G1_{k:02d}",
                "customer_id": f"CUST_VN02_{k:02d}",
                "channel": "email",
                "event_time": BASE_TIME.isoformat(),
                "author_role": "customer",
                "text": wrap_b2b_email(f"Báo giá thì rõ ràng nhưng phản hồi kỹ thuật lần {k} quá chậm.", k),
                "subject": f"Re: Phản hồi tiến độ xử lý đợt {k}",
            })
        else:
            sc_records.append({
                "record_id": f"REC_VN02_G1_{k:02d}",
                "case_id": f"CASE_VN02_G1_{k:02d}",
                "customer_id": f"CUST_VN02_{k:02d}",
                "channel": "service_sheet",
                "event_time": BASE_TIME.isoformat(),
                "author_role": "customer",
                "text": "Báo cáo chi tiết sự cố vận hành vượt giới hạn phân tích. " * 250,
                "product_id": "PROD_01",
            })
    print("Injected VN 02 Group 1 (10 cases).")

## Module 8: Edge-Case Injection — Group 2 (Forwarding, Recurrence & Unknown Identity)

Inject 10 `VN 02` Group 2 scenarios (`DAT 04`, `DAT 05`):
* Cases 1-5: Forwarded emails (`Fwd:`) with recurrence text.
* Cases 6-10: Records with `case_id = "unknown"`, `customer_id = "unknown"`, and `author_role = "unknown"` to verify entity isolation.

In [ ]:
if ENABLE_EDGE_CASES:
    for k in range(1, 11):
        if k <= 5:
            corr_records.append({
                "record_id": f"REC_VN02_G2_{k:02d}",
                "case_id": f"CASE_VN02_G2_{k:02d}",
                "customer_id": f"CUST_VN02_{k:02d}",
                "channel": "email",
                "event_time": (BASE_TIME + timedelta(hours=k)).isoformat(),
                "author_role": "customer",
                "text": wrap_b2b_email("Chuyển tiếp lại phản ánh sự cố lặp lại từ tuần trước.", k),
                "subject": f"Fwd: Sự cố lặp lại lần {k}",
            })
        else:
            sc_records.append({
                "record_id": f"REC_VN02_G2_{k:02d}",
                "case_id": "unknown",
                "customer_id": "unknown",
                "channel": "portal",
                "event_time": BASE_TIME.isoformat(),
                "author_role": "unknown",
                "text": f"Khách hàng chưa định danh phản ánh lỗi kết nối số {k}.",
                "product_id": "PROD_02",
            })
    print("Injected VN 02 Group 2 (10 cases).")

## Module 9: Edge-Case Injection — Group 3 (Missing Rating & Unknown Event Time)

Inject 10 `VN 02` Group 3 scenarios (`DAT 02`, `DAT 06`, `DAT 07`):
* Cases 1-5: Survey records with `event_time = "unknown"` (must be excluded from dated trend series).
* Cases 6-10: Survey records with `rating = None` (must be excluded from valid NPS denominator).

In [ ]:
if ENABLE_EDGE_CASES:
    for k in range(1, 11):
        srv_records.append({
            "record_id": f"REC_VN02_G3_{k:02d}",
            "customer_id": f"CUST_VN02_{k:02d}",
            "channel": "survey",
            "event_time": "unknown" if k <= 5 else (BASE_TIME + timedelta(days=k)).isoformat(),
            "rating": None if k > 5 else (k * 2) % 11,
            "text": f"Khảo sát kiểm thử kỳ thời gian và thiếu điểm đánh giá mẫu {k}.",
        })
    print("Injected VN 02 Group 3 (10 cases).")

## Module 10: Edge-Case Injection — Group 4 (Priority Safety Alerts, Duplicates & Tombstones)

Inject 10 `VN 02` Group 4 scenarios (`DAT 02`, `DAT 05`, `DAT 08`, `ANL 05`):
* Cases 1-5: Urgent safety-related alerts (`quá nhiệt`, `mùi khét`, `cháy nổ`).
* Cases 6-10: Duplicate content rows and `deletion_tombstones` (`REC_SC_00006`..`REC_SC_00010`).

In [ ]:
deletion_tombstones = []

if ENABLE_EDGE_CASES:
    for k in range(1, 11):
        if k <= 5:
            sc_records.append({
                "record_id": f"REC_VN02_G4_{k:02d}",
                "case_id": f"CASE_VN02_G4_{k:02d}",
                "customer_id": f"CUST_VN02_{k:02d}",
                "channel": "ticket",
                "event_time": BASE_TIME.isoformat(),
                "author_role": "customer",
                "text": f"Cảnh báo khẩn: động cơ quá nhiệt, có mùi khét và nguy cơ cháy nổ (mẫu {k}).",
                "product_id": "PROD_03",
            })
        else:
            if sc_records:
                dup_row = dict(sc_records[k - 6])
                dup_row["record_id"] = f"REC_VN02_G4_DUP_{k:02d}"
                sc_records.append(dup_row)
            deletion_tombstones.append(f"REC_SC_{k:05d}")

    print(f"Injected VN 02 Group 4 (10 cases). Tombstones: {deletion_tombstones}")

## Module 11: Package & Export 4 Ready-for-Intake Files

Write `service_case_feedback.csv`, `curated_correspondence.csv`, `survey_responses.csv`, and `batch_manifest.json` (`UTF-8`) into `BASE_DIR`.

In [ ]:
df_sc = pd.DataFrame(sc_records)
df_corr = pd.DataFrame(corr_records)
df_srv = pd.DataFrame(srv_records)

sc_path = os.path.join(BASE_DIR, "service_case_feedback.csv")
corr_path = os.path.join(BASE_DIR, "curated_correspondence.csv")
srv_path = os.path.join(BASE_DIR, "survey_responses.csv")
manifest_path = os.path.join(BASE_DIR, "batch_manifest.json")

df_sc.to_csv(sc_path, index=False, encoding="utf-8")
df_corr.to_csv(corr_path, index=False, encoding="utf-8")
df_srv.to_csv(srv_path, index=False, encoding="utf-8")

now_iso = datetime.now().isoformat(timespec="seconds")
manifest = {
    "batch_id": BATCH_ID,
    "source_version": SOURCE_VERSION,
    "load_mode": LOAD_MODE,
    "cutoff_time": now_iso,
    "extraction_time": now_iso,
    "expected_rows": {
        "service_case_feedback.csv": len(df_sc),
        "curated_correspondence.csv": len(df_corr),
        "survey_responses.csv": len(df_srv),
        "total": len(df_sc) + len(df_corr) + len(df_srv),
    },
    "deletion_tombstones": deletion_tombstones,
    "files": [
        "service_case_feedback.csv",
        "curated_correspondence.csv",
        "survey_responses.csv",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Exported 4 ready files:")
for p in [sc_path, corr_path, srv_path, manifest_path]:
    print(f" - {p}")

## Module 12: Colab Archive & Download

When running inside Google Colab (`/content`), bundle all 4 output files into `ready_for_intake_4_files.zip` and trigger browser download.

In [ ]:
if os.path.exists("/content"):
    !zip -j /content/ready_for_intake_4_files.zip /content/ready_for_intake/*
    try:
        from google.colab import files
        files.download("/content/ready_for_intake_4_files.zip")
    except Exception as e:
        print(f"Download notice: {e}")